In [5]:
from image_load_compression import normalize_batch, load_config

image_urls = ["https://digitalcontent.api.tesco.com/v2/media/ghs/d24f5119-20f7-490d-9bb4-c6a30218ee61/874ca63b-0e66-430b-ae48-686dc755a83d_84541938.jpeg", "https://images3.joy-sourcing.com/product/s1248x1248_jfsintlpro-000-com/t1/4294967296/1179648/61098360190558/603395/68e54737Ecd7a6d5c/4c765c6ab87a3a67.png.webp", "https://media.4rgos.it/i/Argos/7950067_R_Z001A?w=1500&h=880&qlt=70&fmt=webp"]
results = await normalize_batch(image_urls, load_config(), run_id="nb")

In [6]:
import hashlib, json
from datetime import datetime, timezone
from pathlib import Path

def save_results(results, output_dir="output", run_id=None):
    """按 CLI 的约定落盘：图片 + results.jsonl"""
    run_id = run_id or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    target = Path(output_dir) / run_id
    target.mkdir(parents=True, exist_ok=True)

    for r in results:
        if r.image_bytes:
            digest = hashlib.sha256(r.url.encode()).hexdigest()[:16]
            ext = {"JPEG": "jpg"}.get(r.output_format or "", (r.output_format or "bin").lower())
            (target / f"{digest}.{ext}").write_bytes(r.image_bytes)

    with (target / "results.jsonl").open("w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r.to_dict(), ensure_ascii=False) + "\n")
    return target

save_results(results, output_dir="output", run_id="nb")


PosixPath('output/nb')

In [9]:
import pprint

In [10]:
pprint.pprint(results[0])

ImageResult(url='https://digitalcontent.api.tesco.com/v2/media/ghs/d24f5119-20f7-490d-9bb4-c6a30218ee61/874ca63b-0e66-430b-ae48-686dc755a83d_84541938.jpeg',
            status=<Status.BUDGET_UNMET: 'budget_unmet'>,
            original_format='JPEG',
            original_dimensions=(2365, 2365),
            original_size_bytes=229140,
            output_format='JPEG',
            final_dimensions=(2048, 2048),
            final_size_bytes=153010,
            passthrough=False,
            final_quality=40,
            error_layer=None,
            error_detail=None,
            retry_count=0,
            image_bytes=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01'
                        b'\x00\x01\x00\x00\xff\xdb\x00C\x00\x14\x0e\x0f'
                        b'\x12\x0f\r\x14\x12\x10\x12\x17\x15\x14\x18\x1e'
                        b'2!\x1e\x1c\x1c\x1e=,.$2I@LKG@FEPZsbPUmVEFd\x88e'
                        b'mw{\x81\x82\x81N`\x8d\x97\x8c}\x96s~\x81|\xff\xdb\x00'
                 

# image reading 

In [15]:
import os
import base64
from openai import OpenAI

In [18]:
WorkspaceId = "llm-67sm98en2ek0m8jg"
api_key = "sk-ws-H.EXLXHIX.hj5f.MEUCIDPrcQIXlc3_JNiTyOuUr5EHMcHLaJaN1txx4eHpQz95AiEAm82uVi6o3qi_AWZgjpYhwVvwWS9Rm3-T5PdcVU5wQ8s"

In [19]:
import base64

r = results[0]                      # normalize_batch 返回的某个结果对象
mime = "image/jpeg"                 # 或根据 r.output_format 判断

b64 = base64.b64encode(r.image_bytes).decode("utf-8")
data_url = f"data:{mime};base64,{b64}"


In [20]:
client = OpenAI(
    # 若没有配置环境变量，请直接替换为: api_key="sk-xxx"
    api_key=api_key,
    base_url=f"https://{WorkspaceId}.cn-beijing.maas.aliyuncs.com/compatible-mode/v1",
)

completion = client.chat.completions.create(
    model="qwen3.8-flash",
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {
                    # 2. 拼接成标准的 Data URL 格式
                    "url": data_url
                }
            },
            {
                "type": "text",
                "text": "这是什么"
            }
        ]
    }]
)


In [21]:
# 3.7
print(completion.choices[0].message.content)

这是一包 **雀巢 KitKat（奇巧）Chunky 巧克力**，属于 KitKat 系列里更厚、更大块的“Chunky”版本。

从包装上能读出的信息：

- **品牌/产品**：中间白底红字是 Nestlé KitKat 标志，下方蓝框里的 **CHUNKY** 表示这是“厚块/大块”款，比普通四指条更厚实，里面是多层威化夹心＋牛奶巧克力（右下角掰开的那块能看到约四层威化的夹心结构）。
- **规格**：左上角黄色标签 **9x SNACKSIZE BARS**，意思是内含 9 条零食装大小（snacksize）的巧克力条。
- **联名/促销**：右上角带 F1 标志，写着“Formula 1 官方巧克力棒”，并有“WIN VIP MONACO TICKETS…”的抽奖活动（赢取 F1 摩纳哥站 VIP 门票等），两侧的黑白方格旗也是呼应赛车主题。

简单说：**这是一袋 9 条装的 KitKat Chunky 牛奶巧克力威化棒，包装为 F1 赛车联名促销版。**
